# BÁO CÁO KẾT QUẢ VÀ SO SÁNH CÁC MÔ HÌNH PHÂN LOẠI CHỮ SỐ VIẾT TAY

## 1. Mục tiêu

Notebook tổng hợp kết quả các mô hình đã được tinh chỉnh trong những thí nghiệm trước, sau đó xây dựng các mô hình tương ứng bằng **Scikit-learn** để đối chiếu.

Các nhóm mô hình:
- Mô hình tự cài đặt: Logistic Regression, Gaussian Naive Bayes, MLP.
- Mô hình CNN dùng PyTorch: báo cáo riêng vì Scikit-learn không có CNN tương đương trực tiếp.
- Mô hình kết hợp: Soft Voting (MLP + Random Forest + KNN).
- Mô hình Scikit-learn: LogisticRegression, GaussianNB, MLPClassifier, RandomForestClassifier và KNeighborsClassifier.

> Lưu ý: Kết quả tổng hợp ở phần 2 được lấy từ output của các notebook thí nghiệm đã chạy. Phần 4 trở đi là mã chạy lại các mô hình Scikit-learn trên cùng cách chia dữ liệu và tiền xử lý. Thời gian có thể thay đổi theo máy và môi trường chạy.

## 2. Tổng hợp kết quả các best model đã có

### 2.1. Kết quả trên tập test

| Mô hình | Accuracy | Macro F1 / Macro Recall | Train time (s) | Predict time (s) |
|---|---:|---:|---:|---:|
| Logistic Regression (tự cài đặt) | 0.9254 | Macro Recall: 0.9243 | 12.525 | 0.192 |
| Gaussian Naive Bayes (tự cài đặt) | 0.8149 | Macro Recall: 0.8112 | 4.254 | 0.700 |
| MLP (tự cài đặt) | 0.9835 | 0.9834 (Macro Recall) | 98.007 | 0.050 |
| Random Forest (Scikit-learn) | 0.9690 | 0.9687 (Macro Recall) | Đo trong notebook chạy lại | Đo trong notebook chạy lại |
| KNN (Scikit-learn) | 0.9709 | 0.9706 (Macro Recall) | Đo trong notebook chạy lại | Đo trong notebook chạy lại |
| Soft Voting (MLP + RF + KNN) | **0.9848** | **0.9847 (Macro Recall)** | 100.649 | 8.250 |
| CNN (PyTorch, mô hình gốc) | 0.9913 | Chưa có Macro F1 trong output tóm tắt | 516.89 | Chưa ghi nhận |
| MLP (best model, notebook tuning riêng) | 0.9840 | Macro F1: 0.983 | — | — |

**Diễn giải:**
- Trong nhóm mô hình tự cài đặt được đánh giá ở notebook ensemble, MLP có kết quả tốt nhất, vượt Logistic Regression và Gaussian Naive Bayes.
- Soft Voting đạt Accuracy 0.9848, nhỉnh hơn MLP đơn lẻ (0.9835) trong lần chạy được ghi nhận. Tuy nhiên, thời gian dự đoán của ensemble cao hơn đáng kể.
- CNN gốc đạt Accuracy 99.13%, cao nhất trong các kết quả đơn lẻ được tổng hợp ở đây. CNN cần được xem là một nhóm riêng khi so sánh với các thuật toán Scikit-learn dạng vector đặc trưng.
- Không nên so sánh trực tiếp các thời gian ở những lần chạy khác nhau như một benchmark tuyệt đối vì thiết bị, tải hệ thống và cấu hình runtime có thể khác nhau.

### 2.2. Cấu hình tốt nhất được chọn từ các thí nghiệm

| Mô hình | Cấu hình |
|---|---|
| Logistic Regression | `eta=0.03`, `lam=0.001`, `max_epochs=26`, `batch_size=128` |
| Gaussian Naive Bayes | `var_smoothing=0.1`, `batch_size=512` |
| MLP | `sizes=[784, 500, 10]`, `lr=0.05`, `momentum=0.9`, `dropout=0.0`, `weight_decay=0.0001`, `batch_size=128`, `best_epoch=37` |
| CNN | CNN với 421,642 tham số; early stopping tại epoch 13, validation loss tốt nhất ở epoch 7 |
| Soft Voting | Trọng số `[2, 1, 1]` theo thứ tự MLP, RF, KNN |

**Chú ý về tính nhất quán:** kết quả MLP 0.9835 trong notebook ensemble và Accuracy 0.9840 trong notebook tuning MLP đến từ các lần huấn luyện/đánh giá khác nhau. Báo cáo giữ nguyên số liệu của từng nguồn, không gộp thành một kết quả duy nhất.

## 3. Chuẩn bị môi trường và dữ liệu

Các cell dưới đây dùng lại bộ dữ liệu `mnist_train.npz`, `mnist_test.npz`, hàm `split_val` và `scale_flat(mode="standard")` từ project. Hãy chạy notebook trong thư mục project có cấu trúc `src/` và `data/raw/`.

Dữ liệu được chia thành:
- Train: 54,000 mẫu.
- Validation: 6,000 mẫu.
- Test: 10,000 mẫu.

Validation được dùng để chọn cấu hình; tập test chỉ dùng để đánh giá cuối cùng.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "requirements.txt").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(
            "Không tìm thấy project root. Hãy mở notebook từ thư mục project "
            "có requirements.txt, src/ và data/."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
train_data = np.load(DATA_DIR / "mnist_train.npz")
test_data = np.load(DATA_DIR / "mnist_test.npz")

X_train_raw, y_train = train_data["X"], train_data["y"].astype(int)
X_test_raw, y_test = test_data["X"], test_data["y"].astype(int)

from src.preprocessing.scaler import split_val, scale_flat

X_train, X_val, y_train, y_val = split_val(
    X=X_train_raw, y=y_train, val_size=0.1, seed=SEED
)
X_train, X_val, X_test, scaler_tools = scale_flat(
    X_train=X_train,
    X_test=X_test_raw,
    X_val=X_val,
    mode="standard"
)

# Scikit-learn nhận ma trận 2 chiều (n_samples, n_features).
X_train = np.asarray(X_train).reshape(len(X_train), -1)
X_val = np.asarray(X_val).reshape(len(X_val), -1)
X_test = np.asarray(X_test).reshape(len(X_test), -1)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

## 4. Huấn luyện các mô hình Scikit-learn

Để so sánh công bằng hơn, các mô hình bên dưới sử dụng cùng train/validation/test và dữ liệu đã chuẩn hóa. Chọn mô hình bằng Macro F1 trên validation; sau đó fit lại trên train + validation và đánh giá một lần trên test.

Lưu ý:
- `LogisticRegression` của Scikit-learn dùng solver `lbfgs` và regularization `C`; `C` có ý nghĩa nghịch đảo với mức regularization.
- `GaussianNB.var_smoothing` tương ứng gần nhất với tham số smoothing trong Gaussian Naive Bayes, nhưng cách triển khai chi tiết có thể khác.
- `MLPClassifier` là mạng fully-connected của Scikit-learn, không đảm bảo trùng hoàn toàn với MLP tự cài đặt.
- `max_iter` của Scikit-learn không tương đương tuyệt đối với số epoch của mô hình tự cài đặt.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.naive_bayes import GaussianNB as SklearnGaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report

candidate_models = {
    "Sklearn Logistic Regression": [
        SklearnLogisticRegression(C=C, solver="lbfgs", max_iter=300, random_state=SEED)
        for C in [0.1, 1.0, 10.0]
    ],
    "Sklearn GaussianNB": [
        SklearnGaussianNB(var_smoothing=v)
        for v in [1e-3, 1e-2, 1e-1, 2e-1]
    ],
    "Sklearn MLPClassifier": [
        MLPClassifier(
            hidden_layer_sizes=hidden,
            activation="relu",
            solver="adam",
            alpha=alpha,
            batch_size=128,
            learning_rate_init=0.001,
            max_iter=100,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            random_state=SEED
        )
        for hidden, alpha in [((128,), 1e-4), ((256,), 1e-4), ((500,), 1e-4), ((256, 128), 1e-4)]
    ],
    "Sklearn Random Forest": [
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=5,
            random_state=SEED,
            n_jobs=-1
        )
    ],
    "Sklearn KNN": [
        KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=-1)
        for k in [3, 5, 7]
    ],
}

best_sklearn = {}
validation_rows = []

for family, candidates in candidate_models.items():
    best_score = -1
    best_model = None
    best_params = None

    for candidate in candidates:
        start = time.perf_counter()
        candidate.fit(X_train, y_train)
        train_seconds = time.perf_counter() - start

        val_pred = candidate.predict(X_val)
        val_acc = accuracy_score(y_val, val_pred)
        val_f1 = f1_score(y_val, val_pred, average="macro")

        validation_rows.append({
            "Model family": family,
            "Parameters": candidate.get_params(),
            "Validation Accuracy": val_acc,
            "Validation Macro F1": val_f1,
            "Fit time (s)": train_seconds
        })

        if val_f1 > best_score:
            best_score = val_f1
            best_model = candidate
            best_params = candidate.get_params()

    best_sklearn[family] = {
        "model": best_model,
        "params": best_params,
        "validation_macro_f1": best_score
    }

validation_results = pd.DataFrame(validation_rows).sort_values(
    "Validation Macro F1", ascending=False
).reset_index(drop=True)

display(validation_results[[
    "Model family", "Validation Accuracy", "Validation Macro F1", "Fit time (s)"
]])

## 5. Đánh giá best Scikit-learn models trên tập test

Sau khi chọn cấu hình bằng validation, huấn luyện lại từng mô hình trên toàn bộ train + validation. Không sử dụng test để tinh chỉnh tham số.

In [ ]:
X_train_full = np.concatenate([X_train, X_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

sklearn_test_rows = []
sklearn_test_predictions = {}

for family, item in best_sklearn.items():
    final_model = clone(item["model"])

    start = time.perf_counter()
    final_model.fit(X_train_full, y_train_full)
    fit_seconds = time.perf_counter() - start

    start = time.perf_counter()
    test_pred = final_model.predict(X_test)
    predict_seconds = time.perf_counter() - start

    sklearn_test_predictions[family] = test_pred
    sklearn_test_rows.append({
        "Model": family,
        "Accuracy": accuracy_score(y_test, test_pred),
        "Macro Precision": __import__("sklearn.metrics", fromlist=["precision_score"]).precision_score(
            y_test, test_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(y_test, test_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, test_pred, average="macro", zero_division=0),
        "Train time (s)": fit_seconds,
        "Predict time (s)": predict_seconds,
        "Best parameters": item["params"]
    })

sklearn_test_results = pd.DataFrame(sklearn_test_rows).sort_values(
    "Accuracy", ascending=False
).reset_index(drop=True)

display(sklearn_test_results.drop(columns=["Best parameters"]))
print("\\nBest parameters:")
for family, item in best_sklearn.items():
    print(f"\\n{family}: {item['params']}")

## 6. So sánh mô hình tự cài đặt và Scikit-learn

Bảng dưới đây nhập kết quả đã được ghi nhận trong notebook thí nghiệm và nối với kết quả Scikit-learn vừa chạy. Những ô thời gian không có số liệu đáng tin cậy được để trống, không tự suy diễn.

**Quan trọng:** kết quả của mô hình tự cài đặt lấy từ lần chạy trước, trong khi Scikit-learn được chạy lại ở notebook này. Đây là so sánh tham khảo; muốn benchmark nghiêm ngặt cần chạy lại cả hai nhóm trong cùng môi trường, cùng seed và cùng quy trình đo.

In [ ]:
custom_reference = pd.DataFrame([
    {
        "Model": "Custom Logistic Regression",
        "Accuracy": 0.9254,
        "Macro Recall": 0.924316,
        "Macro F1": np.nan,
        "Train time (s)": 12.525,
        "Predict time (s)": 0.192
    },
    {
        "Model": "Custom Gaussian Naive Bayes",
        "Accuracy": 0.8149,
        "Macro Recall": 0.811157,
        "Macro F1": 0.8133,
        "Train time (s)": 4.254,
        "Predict time (s)": 0.700
    },
    {
        "Model": "Custom MLP",
        "Accuracy": 0.9835,
        "Macro Recall": 0.9834,
        "Macro F1": np.nan,
        "Train time (s)": 98.007,
        "Predict time (s)": 0.050
    },
    {
        "Model": "Scikit-learn Random Forest (reference)",
        "Accuracy": 0.9690,
        "Macro Recall": 0.9687,
        "Macro F1": np.nan,
        "Train time (s)": np.nan,
        "Predict time (s)": np.nan
    },
    {
        "Model": "Scikit-learn KNN (reference)",
        "Accuracy": 0.9709,
        "Macro Recall": 0.9706,
        "Macro F1": np.nan,
        "Train time (s)": np.nan,
        "Predict time (s)": np.nan
    },
    {
        "Model": "Soft Voting (MLP + RF + KNN)",
        "Accuracy": 0.9848,
        "Macro Recall": 0.9847,
        "Macro F1": np.nan,
        "Train time (s)": 100.649,
        "Predict time (s)": 8.250
    },
    {
        "Model": "CNN (PyTorch)",
        "Accuracy": 0.9913,
        "Macro Recall": np.nan,
        "Macro F1": np.nan,
        "Train time (s)": 516.89,
        "Predict time (s)": np.nan
    }
])

sklearn_reference = sklearn_test_results[[
    "Model", "Accuracy", "Macro Recall", "Macro F1", "Train time (s)", "Predict time (s)"
]]

all_comparison = pd.concat(
    [custom_reference, sklearn_reference],
    ignore_index=True
).sort_values("Accuracy", ascending=False).reset_index(drop=True)

display(all_comparison)

In [ ]:
# Biểu đồ Accuracy: các mô hình có kết quả test đã ghi nhận
plot_df = all_comparison.dropna(subset=["Accuracy"]).sort_values("Accuracy")

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(plot_df["Model"], plot_df["Accuracy"])
ax.set_xlim(0.75, 1.0)
ax.set_xlabel("Test Accuracy")
ax.set_title("So sánh Accuracy trên tập test")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Nhận xét và kết luận

### 7.1. Nhận xét theo nhóm thuật toán

1. **Logistic Regression:** là mô hình tuyến tính, tốc độ dự đoán nhanh và có khả năng làm baseline tốt. Tuy nhiên, khả năng biểu diễn ranh giới phi tuyến của ảnh chữ số còn hạn chế.
2. **Gaussian Naive Bayes:** có thời gian huấn luyện thấp, nhưng giả định độc lập có điều kiện giữa các đặc trưng chưa phù hợp tốt với pixel ảnh. Kết quả trong thí nghiệm thấp hơn các mô hình còn lại.
3. **MLP:** học được quan hệ phi tuyến giữa các pixel và đạt hiệu năng cao. MLP tự cài đặt trong thí nghiệm đạt Accuracy 0.9835 trên test.
4. **Random Forest và KNN:** là hai baseline phi tuyến từ Scikit-learn. Kết quả tham chiếu lần lượt là 0.9690 và 0.9709.
5. **Soft Voting:** kết hợp xác suất dự đoán của nhiều mô hình có thể cải thiện kết quả, nhưng chi phí suy luận cao hơn do phải chạy nhiều mô hình.
6. **CNN:** khai thác cấu trúc không gian của ảnh thông qua tích chập, phù hợp tự nhiên hơn với dữ liệu ảnh. CNN trong thí nghiệm đạt Accuracy 0.9913.

### 7.2. So sánh mô hình tự cài đặt với Scikit-learn

- Việc so sánh cần tập trung vào cùng họ thuật toán: Custom Logistic Regression ↔ Scikit-learn LogisticRegression; Custom Gaussian Naive Bayes ↔ GaussianNB; Custom MLP ↔ MLPClassifier.
- Không yêu cầu hai thư viện phải cho kết quả giống hệt nhau: thuật toán tối ưu, cách khởi tạo trọng số, regularization, tiêu chí dừng và cách tính xác suất có thể khác nhau.
- Nếu Scikit-learn có Accuracy/Macro F1 cao hơn, điều đó phản ánh cấu hình và implementation cụ thể trong thí nghiệm, không chứng minh thư viện luôn tốt hơn.
- Nếu muốn kết luận chắc chắn về tốc độ, nên chạy lặp nhiều lần (ví dụ 3–5 lần), dùng cùng phần cứng và báo cáo trung bình cùng độ lệch chuẩn.

### 7.3. Kết luận chung

Dựa trên các kết quả hiện có, CNN là lựa chọn có Accuracy cao nhất (99.13%) trong các mô hình đơn được ghi nhận. Soft Voting đạt 98.48%, trong khi MLP đạt khoảng 98.35% ở notebook ensemble. Logistic Regression và Gaussian Naive Bayes có ưu điểm về tính đơn giản hoặc thời gian huấn luyện nhưng độ chính xác thấp hơn.

Kết quả Scikit-learn được tạo khi chạy notebook này sẽ bổ sung một đối chứng cùng dữ liệu cho ba họ thuật toán LR, GNB và MLP, đồng thời cung cấp kết quả chạy lại cho RF và KNN. Cần dựa vào bảng `all_comparison` sau khi chạy hết notebook để đưa ra kết luận định lượng cuối cùng.